# Solution

## Explanation

In this notebook we will work with the [Titanic Data Set](https://www.kaggle.com/c/titanic). We will perform Feature Engineering first and after that, we will train a logistic regression model to predict a classification- survival or deceased model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', None)

## Load Data

Load the 'titanic_train.csv' dataset

In [ ]:
df = pd.read_csv('titanic_train.csv')

### 1. EDA

**Print the first columns**

In [ ]:
df.head()

**Print the shape**

In [ ]:
df.shape

**Print column type**

In [ ]:
df.info()

**Print NaN values**

In [ ]:
df.isna().sum()

**As we have NaN values for 'Age' column, input the values based on the class they belong**

In [ ]:
df.groupby(["Pclass"])["Age"].mean()

In [ ]:
def impute_age(cols):
    Age = cols[0]
    Pclass = cols[1]
    
    if pd.isnull(Age):

        if Pclass == 1:
            return 38

        elif Pclass == 2:
            return 29

        else:
            return 25

    else:
        return Age

In [ ]:
df['Age'] = df[['Age','Pclass']].apply(impute_age,axis=1)

**Check again NaN Values and drop nan values if there are**

In [ ]:
df.isna().sum()

In [ ]:
df.dropna(inplace=True)

### 2. Feature Engineering

**Apply Get Dummies to 'Sex' and 'Embarked' columns and use the argument drop_first=True**

In [ ]:
sex = pd.get_dummies(df['Sex'],drop_first=True)
embark = pd.get_dummies(df['Embarked'],drop_first=True)


# Concat the columns created when applying get_dummmies
df_clean = pd.concat([df,sex,embark],axis=1)

# Visualize data
df_clean.head()

**Create a column with the Title (Dr.,Mr.,Mrs,etc..) from the name as a feature**

*__HINT__ Title always end with a dot (.) may be [this](https://www.w3schools.com/python/ref_string_endswith.asp) can be helpful*


In [ ]:
def extract_title(name):
    for part in name.split():
        if part.endswith('.'):
            return part.replace('.', '')
    return ""

df_clean['title'] = df_clean['Name'].apply(extract_title)

df_clean.head()

**Create a feature with the Cabin letter**

In [ ]:
df_clean["cabin_letter"] = df_clean["Cabin"].apply(lambda x: x[0])

df.head()

**Use get_dummies function to encode the new 2 columns**

In [ ]:
title = pd.get_dummies(df_clean['title'],drop_first=True)
cabin = pd.get_dummies(df_clean['cabin_letter'],drop_first=True)


# Concat the columns created when applying get_dummmies
df_final = pd.concat([df_clean,title,cabin],axis=1)

# Visualize data
df_final.head()

In [ ]:
# Drop 'Sex','Embarked','Name','Ticket','Cabin' columns and be sure all columns are numerical
df_final.drop(['Sex','Embarked','Name','Ticket','Cabin','cabin_letter','title'],axis=1,inplace=True)

df_final.head(10)

### 3. Train Model

**Determine wich columns are predictors (X) and which the target (y)**

In [ ]:
X = df_final.drop(columns=['Survived'],axis=1)

y = df_final['Survived']

**Split the dataset into train and test sets (80/20) and random_state=101**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y,
                                                    test_size=0.20, 
                                                    random_state=101)

**Train the model**

In [ ]:
logmodel = LogisticRegression(max_iter=1000)

logmodel.fit(X_train, y_train)

### Evaluation

In [ ]:
# Get predictions using test dataset
predictions = logmodel.predict(X_test)

# Print report
print(classification_report(y_test,predictions))

In [ ]:
# Generate confusion matrix
conf_matrix = confusion_matrix(y_test, predictions)

# Plotting the confusion matrix
plt.figure(figsize=(10, 6))
sns.heatmap(conf_matrix, annot=True, fmt='g', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

## How to Interpret a Confusion Matrix

A confusion matrix is a visualization tool typically used in supervised learning to measure the performance of a classification algorithm. Here is a guide on how to interpret a confusion matrix:

### Key Components:

- **True Positives (TP)**: The number of correct positive predictions. In the provided matrix, TP is 37.

- **True Negatives (TN)**: The number of correct negative predictions. Here, TN is 13.

- **False Positives (FP)**: The number of incorrect positive predictions (Type I error). In the matrix, FP is 7.

- **False Negatives (FN)**: The number of incorrect negative predictions (Type II error). The matrix shows FN as 4.

### Derived Metrics:

- **Accuracy**: Measures the proportion of true results (both true positives and true negatives) in the total data set. `(TP + TN) / (Total)`

- **Precision**: Indicates the proportion of positive identifications that were actually correct. `TP / (TP + FP)`

- **Recall (Sensitivity)**: Measures the proportion of actual positives that were identified correctly. `TP / (TP + FN)`

- **Specificity**: Measures the proportion of actual negatives that were identified correctly. `TN / (TN + FP)`

- **F1 Score**: The weighted average of Precision and Recall. `2 * (Precision * Recall) / (Precision + Recall)`

### Interpretation Guide for Students:

1. **Identify Elements**: Start by identifying the TP, TN, FP, and FN in your confusion matrix.

2. **Calculate Metrics**: Using the definitions provided, calculate the accuracy, precision, recall, specificity, and F1 score for your model.

3. **Model Performance**: Evaluate the model's performance based on these metrics. High values for precision and recall suggest a better performing model.

4. **Balance Evaluation**: Consider the trade-off between precision and recall. An F1 score can help you understand the balance between these two metrics.

By following this guide, you can determine the efficacy of your classification model and understand the implications of its performance in practical applications.
